Selecting a team, with budget and position constaints, while maximizing some expected metric, can be considered as a constaint maximization problem (specifically, a Mixed Integer Linear Program).

In [36]:
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger("util").setLevel(logging.ERROR)

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy.optimize import minimize

df = pd.read_csv('/kaggle/input/datasets/calvinrostanto/fantasy-premier-league-2025-2026/fpl_player_statistics.csv')

In [37]:
#Used to solve the constraint maximization
!pip install cvxpy

Now, we need a metric to maximize. I combine total points and form, like this:

$$ \text{ $metric = form * w1 + totalpoints * w2$} $$

Earlier, I used a base metric
$$ \text{ $w1 = totalpoints.max(), w2 = form.max()$} $$

But, these weights can also be learned, by looking at last season's per gameweek data **(this data is obtained from vaastav's FPL dataset [Link](https://github.com/vaastav/Fantasy-Premier-League/tree/master)** 

We aim to use total points and form at current gameweek to maximize the accuracy of top-20 predictions for average points per game in the remaining games.

In [38]:
gw_level_points = pd.read_csv('/kaggle/input/datasets/teamaker/vaastav-gw-level-player-data/merged_gw_24_25.csv')

cols_to_keep = ['name', 'total_points', 'GW']
df_gw = gw_level_points[cols_to_keep]

num_gws_before_wildcard = 32
num_future_gws = 6

df_gw = df_gw[df_gw['GW'] <= (num_gws_before_wildcard + num_future_gws)]

df_gw['total_points_so_far'] = df_gw.groupby('name')['total_points'].cumsum()

df_gw['form'] = (df_gw.groupby('name')['total_points']
  .transform(lambda x: x.rolling(6, min_periods=6).sum()/6.0)
)

#Get a single dataframe containing form, total_points for each player and average points in remaining games
future = df_gw[(df_gw["GW"] >= num_gws_before_wildcard+1) & (df_gw["GW"] <= num_gws_before_wildcard+num_future_gws)]
df_players = df_gw[df_gw["GW"] == num_gws_before_wildcard].copy()
avg_points = future.groupby("name")["total_points"].mean()
df_players["avg_next_points"] = df_players["name"].map(avg_points)

In [39]:
#w1 and w2 : between 0 and 1
def optimal_weights(X_train, X_val):

    best = (-1, None)
    K = 20
    
    X_train['actual_rank'] = y_train['avg_next_points'].rank(ascending=False).astype(int)
    true_topK = set(X_train.nsmallest(K, "actual_rank").index)
        
    for w1 in np.linspace(0, 1, 40):
        if w1 < 0.1:
            continue
        w2 = 1 - w1
        score = w1 * X_train['form_norm'] + w2 * X_train['total_points_so_far_norm']
        X_train["score"] = score
        X_train['rank_metric'] = X_train['score'].rank(ascending=False).astype(int)
        pred_topK = set(X_train.nsmallest(K, "rank_metric").index)
        
        acc = len(true_topK & pred_topK) / K
        
        if acc > best[0]:
            best = (acc, (w1, w2))
        
    print (f"Best w1 : {best[1][0]:.2f} and w2 : {best[1][1]:.2f}")
    
    pred_topK_base = set(X_train.nsmallest(K, "rank_metric_base").index)
    train_acc_base = len(true_topK & pred_topK_base) / K        
    
    X_val['actual_rank'] = y_val['avg_next_points'].rank(ascending=False).astype(int)
    true_topK = set(X_val.nsmallest(K, "actual_rank").index)
        
    w1 = best[1][0]
    w2 = best[1][1]
    score = w1 * X_val['form_norm'] + w2 * X_val['total_points_so_far_norm']
    
    X_val["score"] = score
    X_val['rank_metric'] = X_val['score'].rank(ascending=False).astype(int)
    pred_topK = set(X_val.nsmallest(K, "rank_metric").index)
    acc = len(true_topK & pred_topK) / K
    
    pred_topK_base = set(X_val.nsmallest(K, "rank_metric_base").index)
    acc_base = len(true_topK & pred_topK_base) / K  

    return (acc, acc_base, best[0], train_acc_base)

In [40]:
from sklearn.model_selection import train_test_split

df_players = df_players[~df_players['form'].isna()]
df_players.index = range(df_players.shape[0])

#normalize
def preprocessing(X_train, X_val):
    means = {}
    stds = {}
    for col in ['form', 'total_points_so_far']:
        means[col] = X_train[col].mean()
        stds[col] = X_train[col].std()
        X_train[col + '_norm'] = (X_train[col] - means[col])/stds[col]
        X_val[col + '_norm'] = (X_val[col] - means[col])/stds[col]
    
    X_train['metric'] = X_train['total_points_so_far_norm'] * X_train['form_norm'].max() + X_train['form_norm'] * X_train['total_points_so_far_norm'].max()
    X_val['metric'] = X_val['total_points_so_far_norm'] * X_train['form_norm'].max() + X_val['form_norm'] * X_train['total_points_so_far_norm'].max()
    
    for df_foo in [X_train, X_val]:
        df_foo['rank_metric_base'] = df_foo['metric'].rank(ascending=False).astype(int)

    return X_train, X_val


cols_x = ['total_points_so_far', 'form', 'name']
cols_y = ['avg_next_points']

X = df_players[cols_x]
y = df_players[cols_y]

from sklearn.model_selection import KFold

avg_val_acc = 0
avg_val_acc_base = 0
avg_train_acc = 0
avg_train_acc_base = 0

#2-fold split since accuracy of top predictions is dependent on dataset size, so keep size of train and val the same
kf = KFold(n_splits=2, shuffle=True, random_state=42)

k = 0
for train_index, test_index in kf.split(X):

    k+=1
    print (f"Fold number : {k}")

    X_train, X_val = X.iloc[train_index], X.iloc[test_index]
    y_train, y_val = y.iloc[train_index], y.iloc[test_index]

    X_train, X_val = preprocessing(X_train, X_val)
        
    val_acc, val_acc_base, train_acc, train_acc_base = optimal_weights(X_train, X_val)
    avg_val_acc += val_acc/2
    avg_val_acc_base += val_acc_base/2
    avg_train_acc += train_acc/2
    avg_train_acc_base += train_acc_base/2
    
cols_x.remove('name')

#compare with base metric to see if learning improves accuracy
print (f"Avg train accuracy : {avg_train_acc:.2f} and for base : {avg_train_acc_base:.2f}")
print (f"Avg val accuracy : {avg_val_acc:.2f} and for base : {avg_val_acc_base:.2f}")

#Now, find optimal weights to use by using the entire dataset
#bit hacky
X_train, X_val = preprocessing(df_players[cols_x], df_players[cols_x])
val_acc, val_acc_base, train_acc, train_acc_base = optimal_weights(X_train, X_val)

print (f"Train accuracy for entire dataset : {train_acc:.2f} and for base : {train_acc_base:.2f}")

Fold number : 1
Best w1 : 0.64 and w2 : 0.36
Fold number : 2
Best w1 : 0.69 and w2 : 0.31
Avg train accuracy : 0.35 and for base : 0.28
Avg val accuracy : 0.33 and for base : 0.28
Best w1 : 0.85 and w2 : 0.15
Train accuracy for entire dataset : 0.25 and for base : 0.15


As we can see, optimizing the weights improves accuracy of top-20 predictions, compared to range-based weights. Our final weights after optimizing on the entire 2024-25 dataset are :
                                    $$ \text{ $w1 = 0.85, w2 = 0.15$} $$

On average, 
* **Our model predicts 5 out of the top 20 performing players for the final 5 games of the season**
* **The model values form significantly more than total points so far in making these predictions**

Nice. Let's use those weights to define our metric, which we can maximimize using constraint maximization.

In [47]:
means = {}
stds = {}
for col in ['form', 'total_points']:
    means[col] = df_players[col].mean()
    stds[col] = df_players[col].std()
    df[col + '_norm'] = (df[col] - means[col])/stds[col]

df['metric'] = df['form_norm'] * 0.85 + df['total_points_norm'] * 0.15

In [49]:
df.index = range(df.shape[0])

In [50]:
import cvxpy as cp

values = df['metric']
costs = df['now_cost'].tolist()
positions = df['position_name'].tolist()
budget = 84.1                   # Budget constraint - select only 11 players and select 
                                # the 4 substitutes with minimum value players
                                 
n = len(values)            
max_items = 11

# Decision variables
x = cp.Variable(n, boolean=True)  # Binary variables: 1 if player is selected, 0 otherwise

# Objective function
objective = cp.Maximize(cp.sum(cp.multiply(values, x)))

fwd_indices = df[df['position_name']=='FWD'].index.tolist()
mid_indices = df[df['position_name']=='MID'].index.tolist()
def_indices = df[df['position_name']=='DEF'].index.tolist()
gkp_indices = df[df['position_name']=='GKP'].index.tolist()

# Constraint: total cost ≤ budget, number of players in each position, and total number of players selected
constraints = [
    cp.sum(cp.multiply(costs, x)) <= budget,
    cp.sum(x) <= max_items,
    cp.sum(x[fwd_indices]) <= 3,
    cp.sum(x[mid_indices]) <= 4,
    cp.sum(x[def_indices]) <= 4,
    cp.sum(x[gkp_indices]) == 1
    ]

problem = cp.Problem(objective, constraints)
problem.solve()

116.70516647839494

In [55]:
#print the selected players after GW 32 for 2025-26 season
df[['player_name', 'position_name', 'now_cost', 'form', 'total_points', 'metric']][x.value.astype(bool)].sort_values(by='position_name')

,player_name,position_name,now_cost,form,total_points,metric
2,Gabriel Gabriel,DEF,7.1,2.0,177,10.384654
9,Marc Guéhi,DEF,5.1,5.0,150,10.289352
52,Neco N.Williams,DEF,4.8,8.7,115,10.074871
0,Erling Haaland,FWD,14.5,0.7,199,11.013304
4,Igor Thiago Thiago,FWD,7.3,6.3,168,11.941682
8,Jarrod Bowen,FWD,7.6,5.3,157,10.832959
37,Caoimhín Kelleher,GKP,4.8,5.7,122,9.029684
1,Bruno B.Fernandes,MID,10.3,5.7,193,13.078422
3,Antoine Semenyo,MID,8.2,0.7,176,9.701741
5,Declan Rice,MID,7.2,2.0,167,9.814409


In [58]:
#Overall top 10 performing player prediction after GW 32 for 2025-26 season
df[['player_name', 'position_name', 'now_cost', 'form', 'total_points', 'metric']].iloc[:10].sort_values(by='metric', ascending=False)

,player_name,position_name,now_cost,form,total_points,metric
1,Bruno B.Fernandes,MID,10.3,5.7,193,13.078422
4,Igor Thiago Thiago,FWD,7.3,6.3,168,11.941682
0,Erling Haaland,FWD,14.5,0.7,199,11.013304
8,Jarrod Bowen,FWD,7.6,5.3,157,10.832959
7,Harry Wilson,MID,6.0,4.7,157,10.544088
2,Gabriel Gabriel,DEF,7.1,2.0,177,10.384654
9,Marc Guéhi,DEF,5.1,5.0,150,10.289352
5,Declan Rice,MID,7.2,2.0,167,9.814409
3,Antoine Semenyo,MID,8.2,0.7,176,9.701741
6,João Pedro João Pedro,FWD,7.7,1.0,166,9.275932


I use the base metric at the start of the season, and optimized weight metric for wildcard/free hit. I'm trying to see if the metric can be better defined as the score output of a binary classification model, and/or using other variables. In addition, adding data for the past 4-5 seasons should make the metric more stable. Let me know your thoughts in the comments!